In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

# ============================================
# CONFIGURATION
# ============================================
N_TRANSACTIONS = 100_000  # 100,000 transactions
N_CUSTOMERS = 5_000      # 5,000 unique customers
N_PRODUCTS = 200         # 200 unique products
N_STORES = 4             # 4 stores
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2024, 12, 31)  # 2 years of data

# ============================================
# FILE 1: STORE_LOCATIONS.CSV
# ============================================
stores_data = {
    'store_id': [101, 102, 103, 104],
    'store_name': ['Downtown Plaza', 'Westfield Mall', 'Eastside Market', 'Northpoint Center'],
    'region': ['North', 'South', 'East', 'West'],
    'manager': ['Alice Johnson', 'Bob Smith', 'Carol White', 'David Brown'],
    'sq_footage': [45000, 62000, 38000, 55000],
    'opening_date': ['2018-03-15', '2019-07-22', '2020-01-10', '2017-11-05']
}

stores_df = pd.DataFrame(stores_data)
stores_df.to_csv('store_locations.csv', index=False)
print(f"✓ store_locations.csv created: {len(stores_df)} stores")

# ============================================
# FILE 2: PRODUCT_MASTER.CSV
# ============================================
categories = ['Electronics', 'Clothing', 'Home & Garden']
category_weights = [0.35, 0.40, 0.25]  # Electronics sells more

products = []
product_id = 1001

for i in range(N_PRODUCTS):
    cat = np.random.choice(categories, p=category_weights)
    
    # Product names by category
    if cat == 'Electronics':
        name = random.choice(['Laptop', 'Phone', 'Tablet', 'Headphones', 'Smartwatch', 
                              'Camera', 'Speaker', 'Monitor', 'Keyboard', 'Mouse']) + f' Model-{i+1}'
        cost = random.uniform(50, 800)
    elif cat == 'Clothing':
        name = random.choice(['T-Shirt', 'Jeans', 'Jacket', 'Sneakers', 'Dress',
                              'Sweater', 'Shorts', 'Hat', 'Scarf', 'Socks']) + f' Style-{i+1}'
        cost = random.uniform(8, 120)
    else:  # Home & Garden
        name = random.choice(['Sofa', 'Lamp', 'Chair', 'Table', 'Rug',
                              'Planter', 'Cushion', 'Curtain', 'Shelf', 'Vase']) + f' Design-{i+1}'
        cost = random.uniform(15, 600)
    
    products.append({
        'product_id': product_id,
        'product_name': name,
        'category': cat,
        'unit_cost': round(cost, 2),
        'launch_date': (START_DATE + timedelta(days=random.randint(0, 365))).strftime('%Y-%m-%d')
    })
    product_id += 1

products_df = pd.DataFrame(products)
products_df.to_csv('product_master.csv', index=False)
print(f"✓ product_master.csv created: {len(products_df)} products")

# ============================================
# FILE 3: CUSTOMERS.CSV
# ============================================
segments = ['Premium', 'Standard', 'Budget']
segment_weights = [0.20, 0.55, 0.25]

customers = []
for i in range(N_CUSTOMERS):
    signup = START_DATE + timedelta(days=random.randint(0, 730))
    age = random.randint(18, 75)
    
    # Segment correlates with age and signup date
    if age > 50 and signup.year <= 2023:
        seg = np.random.choice(segments, p=[0.50, 0.40, 0.10])
    elif age < 30:
        seg = np.random.choice(segments, p=[0.10, 0.40, 0.50])
    else:
        seg = np.random.choice(segments, p=segment_weights)
    
    customers.append({
        'customer_id': 5001 + i,
        'age': age,
        'signup_date': signup.strftime('%Y-%m-%d'),
        'segment': seg,
        'region': random.choice(['North', 'South', 'East', 'West'])
    })

customers_df = pd.DataFrame(customers)
customers_df.to_csv('customers.csv', index=False)
print(f"✓ customers.csv created: {len(customers_df)} customers")

# ============================================
# FILE 4: SALES_TRANSACTIONS.CSV (THE BIG ONE)
# ============================================
# Pre-compute for speed
store_ids = stores_df['store_id'].tolist()
product_ids = products_df['product_id'].tolist()
customer_ids = customers_df['customer_id'].tolist()

# Create price lookup from product cost (markup 1.5x to 3x)
price_lookup = {}
for _, row in products_df.iterrows():
    price_lookup[row['product_id']] = round(row['unit_cost'] * random.uniform(1.5, 3.0), 2)

transactions = []
transaction_id = 1

# Generate transactions with realistic patterns
date_range = (END_DATE - START_DATE).days

for _ in range(N_TRANSACTIONS):
    # Date with seasonality (more sales in Nov-Dec, June-July)
    day_offset = random.randint(0, date_range)
    date = START_DATE + timedelta(days=day_offset)
    
    # Boost sales in holiday seasons
    month = date.month
    if month in [11, 12]:  # Holiday
        if random.random() > 0.6:  # 40% chance to regenerate for more density
            day_offset = random.randint(300, 365) if month == 11 else random.randint(330, 395)
            date = START_DATE + timedelta(days=min(day_offset, date_range))
    elif month in [6, 7]:  # Summer
        if random.random() > 0.7:
            day_offset = random.randint(150, 210)
            date = START_DATE + timedelta(days=day_offset)
    
    # Select customer, then prefer their region's store
    customer = customers_df[customers_df['customer_id'] == random.choice(customer_ids)].iloc[0]
    preferred_store = random.choice(store_ids)  # Simplified; could weight by region
    
    product_id = random.choice(product_ids)
    unit_price = price_lookup[product_id]
    
    # Quantity correlates with price (cheaper = more units)
    if unit_price < 50:
        qty = random.randint(1, 5)
    elif unit_price < 200:
        qty = random.randint(1, 3)
    else:
        qty = random.randint(1, 2)
    
    # Apply occasional discount (10% of transactions)
    discount = 0
    if random.random() < 0.10:
        discount = random.choice([0.05, 0.10, 0.15, 0.20])
    
    revenue = round(unit_price * qty * (1 - discount), 2)
    
    transactions.append({
        'transaction_id': transaction_id,
        'date': date.strftime('%Y-%m-%d'),
        'store_id': preferred_store,
        'product_id': product_id,
        'customer_id': customer['customer_id'],
        'quantity': qty,
        'unit_price': round(unit_price, 2),
        'discount_pct': round(discount * 100, 0),
        'revenue': revenue
    })
    
    transaction_id += 1
    
    # Progress indicator
    if transaction_id % 20_000 == 0:
        print(f"  Generated {transaction_id:,} transactions...")

sales_df = pd.DataFrame(transactions)
sales_df.to_csv('sales_transactions.csv', index=False)
print(f"✓ sales_transactions.csv created: {len(sales_df):,} transactions")

# ============================================
# VALIDATION SUMMARY
# ============================================
print("\n" + "="*50)
print("DATA GENERATION COMPLETE")
print("="*50)
print(f"\nFiles created:")
print(f"  • store_locations.csv    : {len(stores_df):,} rows")
print(f"  • product_master.csv       : {len(products_df):,} rows")
print(f"  • customers.csv            : {len(customers_df):,} rows")
print(f"  • sales_transactions.csv   : {len(sales_df):,} rows")
print(f"\nDate range: {sales_df['date'].min()} to {sales_df['date'].max()}")
print(f"Total revenue: ${sales_df['revenue'].sum():,.2f}")
print(f"Unique customers in transactions: {sales_df['customer_id'].nunique():,}")
print(f"Unique products sold: {sales_df['product_id'].nunique():,}")

✓ store_locations.csv created: 4 stores
✓ product_master.csv created: 200 products
✓ customers.csv created: 5000 customers
  Generated 20,000 transactions...
  Generated 40,000 transactions...
  Generated 60,000 transactions...
  Generated 80,000 transactions...
  Generated 100,000 transactions...
✓ sales_transactions.csv created: 100,000 transactions

DATA GENERATION COMPLETE

Files created:
  • store_locations.csv    : 4 rows
  • product_master.csv       : 200 rows
  • customers.csv            : 5,000 rows
  • sales_transactions.csv   : 100,000 rows

Date range: 2023-01-01 to 2024-12-31
Total revenue: $87,377,107.13
Unique customers in transactions: 5,000
Unique products sold: 200


In [ ]:
import pandas as pd

sales = pd.read_csv('sales_transactions.csv')
stores = pd.read_csv('store_locations.csv')
products =pd.read_csv('product_master.csv')
customer = pd.read_csv('customers.csv')

# Check: Every store_id in sales exist in stores?
print(sales['store_id'].isin(stores['store_id']).all())

# Check: Every product_id in sales exists in products?
print(sales['product_id'].isin(sales['product_id']).all())

# Check: Revenue = unit_price * quantity * (1 - discount/100)?
sales['calc_revenue'] = sales['unit_price'] * sales['quantity'] * (1 - sales['discount_pct']/100)
print((sales['revenue'].round(2) == sales['calc_revenue'].round(2)).all())

True
True
False
False


In [ ]:
sales['calc_revenue'] = sales['unit_price'] * sales['quantity'] * (1 - sales['discount_pct']/100)

# Checking if the values are close enough (within 1 cent)
matches = np.isclose(sales['revenue'], sales['calc_revenue'], atol=0.01)
print(matches.all())

# See how many don't match (if any)
print(f"Non--matches: {(~matches).sum()}")```````````````
print(sales[~matches][['revenue', 'calc_revenue']].head())

True
Non--matches: 0
Empty DataFrame
Columns: [revenue, calc_revenue]
Index: []
